## Import

In [96]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

In [97]:
df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
docs[:5]
classes = list(df["gen"])

## Pre-caculate Embeddings

Shorten run time once calculated

In [98]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches: 100%|██████████| 32/32 [00:12<00:00,  2.54it/s]


## Preventing Stochastic Behavior
Reduce dimenstion (size of embeddings).

Also allows for reproduction every time the model is run.

In [45]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

## Controlling Number of Topics
Using HDBSCAN, we can merge topics **after** creation.

In [46]:
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)


## Improving default representation
- Remove stopwords
- Ignore infrequent words
- Increase the n-gram range (to 2) 

In [47]:
vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))


## Training

In [118]:
topic_model = BERTopic(

    # Pipeline models
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,

    # Hyperparameters
    top_n_words=10,
    verbose=True,

    # General parameters
    calculate_probabilities=True,
    language="english"
)

# Train model
topics, probs = topic_model.fit_transform(docs, embeddings)
topic_per_class = topic_model.topics_per_class(docs, classes=classes)
topics_over_time = topic_model.topics_over_time(docs, classes)

# Show topics
topic_model.get_topic_info()

2025-01-24 15:42:29,097 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-01-24 15:42:32,050 - BERTopic - Dimensionality - Completed ✓
2025-01-24 15:42:32,051 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-01-24 15:42:32,104 - BERTopic - Cluster - Completed ✓
2025-01-24 15:42:32,109 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-01-24 15:42:32,220 - BERTopic - Representation - Completed ✓
10it [00:00, 10.31it/s]
10it [00:00, 24.10it/s]


,Topic,Count,Name,Representation,Representative_Docs
0,-1,197,-1_gnome_gnomes_mushrooms_round,"[gnome, gnomes, mushrooms, round, multiplier, ...",[Every round shows a pair of gnomes. The gnome...
1,0,158,0_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...
2,1,114,1_points_gnome_gnomes_colour,"[points, gnome, gnomes, colour, group, value, ...",[Gnomes will provide points varying from 0-9 b...
3,2,103,2_points_blue_colours_pink,"[points, blue, colours, pink, purple, pattern,...","[There are two themes of colours, purple, pink..."
4,3,99,3_mushrooms_gnome_gnomes_colour,"[mushrooms, gnome, gnomes, colour, try, gives,...",[There's a pattern when it comes to the gnomes...
5,4,84,4_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points, gnomes,...",[On the screen you will be presented with two ...
6,5,68,5_mushrooms_colours_mushroom_change,"[mushrooms, colours, mushroom, change, color, ...",[Pay attention to the colours as certain colo...
7,6,55,6_hat_gnome_gnomes_tall,"[hat, gnome, gnomes, tall, short, hats, points...",[Choose the gnome with a short yellow hat or t...
8,7,34,7_hats_tall_hat_tall hats,"[hats, tall, hat, tall hats, short, taller, go...","[Do your best, seems pretty random and difficu..."
9,8,27,8_keys_breaks_fingers_just,"[keys, breaks, fingers, just, game, hand, make...","[As each gnome appears, tap S for the left gno..."


## Vizualisation

In [91]:
topic_model.visualize_topics()

In [53]:
topic_model.visualize_distribution(probs[10], min_probability=0.015)

In [54]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [88]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

100%|██████████| 11/11 [00:00<00:00, 123.18it/s]

.
├─hat_hats_tall_gnome_short
│    ├─hat_hats_tall_short_gnome
│    │    ├─■──hats_tall_hat_tall hats_short ── Topic: 7
│    │    └─■──hat_gnome_gnomes_tall_short ── Topic: 6
│    └─■──hat_mushrooms_hats_tall_modifier ── Topic: 10
└─basket_mushrooms_red_gnomes_yellow
     ├─basket_mushrooms_red_gnomes_yellow
     │    ├─basket_red_yellow_gnomes_mushrooms
     │    │    ├─■──forest_gnomes forest_mushrooms_green_gnomes ── Topic: 9
     │    │    └─basket_red_yellow_gnomes_red basket
     │    │         ├─■──basket_red_yellow_mushrooms_gnomes ── Topic: 0
     │    │         └─■──basket_red_baskets_yellow_points ── Topic: 4
     │    └─mushrooms_points_gnome_colour_colours
     │         ├─mushrooms_gnome_points_colours_colour
     │         │    ├─mushrooms_gnome_colours_gnomes_colour
     │         │    │    ├─■──mushrooms_colours_mushroom_change_color ── Topic: 5
     │         │    │    └─■──mushrooms_gnome_gnomes_colour_try ── Topic: 3
     │         │    └─points_gnome_colour_colours

In [33]:
topic_model.visualize_documents(docs)

In [35]:
topic_model.visualize_heatmap()

In [40]:
topic_model.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,Consider each round carefully as it appears th...,2,2_points_blue_colours_pink,"[points, blue, colours, pink, purple, pattern,...","[There are two themes of colours, purple, pink...",points - blue - colours - pink - purple - patt...,1.000000,False
1,"If the multiplier is 5, then choose the pointy...",-1,-1_gnome_gnomes_mushrooms_round,"[gnome, gnomes, mushrooms, round, multiplier, ...",[Every round shows a pair of gnomes. The gnome...,gnome - gnomes - mushrooms - round - multiplie...,0.670309,False
2,The game is fantastic but requires you to thin...,8,8_keys_breaks_fingers_just,"[keys, breaks, fingers, just, game, hand, make...","[As each gnome appears, tap S for the left gno...",keys - breaks - fingers - just - game - hand -...,0.227426,False
3,My advice would be to pay attention to the col...,1,1_points_gnome_gnomes_colour,"[points, gnome, gnomes, colour, group, value, ...",[Gnomes will provide points varying from 0-9 b...,points - gnome - gnomes - colour - group - val...,1.000000,False
4,"Do not let the timer worry you, but make sure ...",8,8_keys_breaks_fingers_just,"[keys, breaks, fingers, just, game, hand, make...","[As each gnome appears, tap S for the left gno...",keys - breaks - fingers - just - game - hand -...,1.000000,False
...,...,...,...,...,...,...,...,...
995,Try to match the gnome colours to the yellow ...,0,0_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...,basket - red - yellow - mushrooms - gnomes - r...,1.000000,False
996,based on the colours of the gnomes - keep chan...,9,9_forest_gnomes forest_mushrooms_green,"[forest, gnomes forest, mushrooms, green, gnom...","[There are two forests. Green, orange, yellow,...",forest - gnomes forest - mushrooms - green - g...,1.000000,False
997,"hello, there are two different colors of baske...",0,0_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...,basket - red - yellow - mushrooms - gnomes - r...,0.102782,False
998,"The taller gnomes perform better, yellow is th...",6,6_hat_gnome_gnomes_tall,"[hat, gnome, gnomes, tall, short, hats, points...",[Choose the gnome with a short yellow hat or t...,hat - gnome - gnomes - tall - short - hats - p...,0.303229,False


In [51]:
topic_model.visualize_topics_per_class(topic_per_class)

In [67]:
topic_model.visualize_topics_over_time(topics_over_time)

In [86]:
topic_model.get_topic(7)

[('hats', 0.13831835123478242),
 ('tall', 0.08147863832976292),
 ('hat', 0.07959362376918742),
 ('tall hats', 0.04254449357971294),
 ('short', 0.04193779586168998),
 ('taller', 0.037918844506754014),
 ('good', 0.03511173865885397),
 ('try', 0.032535475916732705),
 ('best', 0.0319438587507837),
 ('better', 0.030885566526946167)]

## Data saving

In [83]:
import csv

with open(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_topic_value.csv"), "w", newline="") as file:
    write = csv.writer(file)
    for topic in topics:
        write.writerow([topic])

## Topic reduction

In [107]:
a = topic_model

a.reduce_topics(docs, nr_topics = 5)

a.get_topic_info()

2025-01-24 15:26:24,481 - BERTopic - Topic reduction - Reducing number of topics
2025-01-24 15:26:25,037 - BERTopic - Topic reduction - Reduced number of topics from 13 to 5


,Topic,Count,Name,Representation,Representative_Docs
0,-1,197,-1_gnome_gnomes_mushrooms_colour,"[gnome, gnomes, mushrooms, colour, round, brow...",[Every round shows a pair of gnomes. The gnome...
1,0,639,0_basket_gnomes_mushrooms_red,"[basket, gnomes, mushrooms, red, gnome, yellow...",[There are 8 gnomes of different colours. The ...
2,1,103,1_points_blue_colours_pink,"[points, blue, colours, pink, purple, pattern,...","[There are two themes of colours, purple, pink..."
3,2,34,2_hats_hat_tall_short,"[hats, hat, tall, short, tall hats, good, tall...","[Do your best, seems pretty random and difficu..."
4,3,27,3_keys_breaks_game_just,"[keys, breaks, game, just, fingers, make, hand...","[As each gnome appears, tap S for the left gno..."


In [110]:
a.visualize_heatmap()

## TEST

In [84]:
topic_model.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,Consider each round carefully as it appears th...,2,2_points_blue_colours_pink,"[points, blue, colours, pink, purple, pattern,...","[There are two themes of colours, purple, pink...",points - blue - colours - pink - purple - patt...,1.000000,False
1,"If the multiplier is 5, then choose the pointy...",-1,-1_gnome_gnomes_mushrooms_round,"[gnome, gnomes, mushrooms, round, multiplier, ...",[Every round shows a pair of gnomes. The gnome...,gnome - gnomes - mushrooms - round - multiplie...,0.670309,False
2,The game is fantastic but requires you to thin...,8,8_keys_breaks_fingers_just,"[keys, breaks, fingers, just, game, hand, make...","[As each gnome appears, tap S for the left gno...",keys - breaks - fingers - just - game - hand -...,0.227426,False
3,My advice would be to pay attention to the col...,1,1_points_gnome_gnomes_colour,"[points, gnome, gnomes, colour, group, value, ...",[Gnomes will provide points varying from 0-9 b...,points - gnome - gnomes - colour - group - val...,1.000000,False
4,"Do not let the timer worry you, but make sure ...",8,8_keys_breaks_fingers_just,"[keys, breaks, fingers, just, game, hand, make...","[As each gnome appears, tap S for the left gno...",keys - breaks - fingers - just - game - hand -...,1.000000,False
...,...,...,...,...,...,...,...,...
995,Try to match the gnome colours to the yellow ...,0,0_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...,basket - red - yellow - mushrooms - gnomes - r...,1.000000,False
996,based on the colours of the gnomes - keep chan...,9,9_forest_gnomes forest_mushrooms_green,"[forest, gnomes forest, mushrooms, green, gnom...","[There are two forests. Green, orange, yellow,...",forest - gnomes forest - mushrooms - green - g...,1.000000,False
997,"hello, there are two different colors of baske...",0,0_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...,basket - red - yellow - mushrooms - gnomes - r...,0.102782,False
998,"The taller gnomes perform better, yellow is th...",6,6_hat_gnome_gnomes_tall,"[hat, gnome, gnomes, tall, short, hats, points...",[Choose the gnome with a short yellow hat or t...,hat - gnome - gnomes - tall - short - hats - p...,0.303229,False


topic reduction

In [120]:
new_topics = topic_model.reduce_outliers(docs, topics)

100%|██████████| 1/1 [00:00<00:00,  4.97it/s]


,Topic,Count,Name,Representation,Representative_Docs
0,-1,197,-1_gnome_gnomes_mushrooms_round,"[gnome, gnomes, mushrooms, round, multiplier, ...",[Every round shows a pair of gnomes. The gnome...
1,0,158,0_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...",[There are eight gnomes of different colours (...
2,1,114,1_points_gnome_gnomes_colour,"[points, gnome, gnomes, colour, group, value, ...",[Gnomes will provide points varying from 0-9 b...
3,2,103,2_points_blue_colours_pink,"[points, blue, colours, pink, purple, pattern,...","[There are two themes of colours, purple, pink..."
4,3,99,3_mushrooms_gnome_gnomes_colour,"[mushrooms, gnome, gnomes, colour, try, gives,...",[There's a pattern when it comes to the gnomes...
5,4,84,4_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points, gnomes,...",[On the screen you will be presented with two ...
6,5,68,5_mushrooms_colours_mushroom_change,"[mushrooms, colours, mushroom, change, color, ...",[Pay attention to the colours as certain colo...
7,6,55,6_hat_gnome_gnomes_tall,"[hat, gnome, gnomes, tall, short, hats, points...",[Choose the gnome with a short yellow hat or t...
8,7,34,7_hats_tall_hat_tall hats,"[hats, tall, hat, tall hats, short, taller, go...","[Do your best, seems pretty random and difficu..."
9,8,27,8_keys_breaks_fingers_just,"[keys, breaks, fingers, just, game, hand, make...","[As each gnome appears, tap S for the left gno..."


[2,
 -1,
 8,
 1,
 8,
 5,
 11,
 2,
 -1,
 8,
 1,
 -1,
 -1,
 3,
 7,
 6,
 3,
 -1,
 11,
 5,
 6,
 3,
 -1,
 -1,
 3,
 1,
 2,
 1,
 -1,
 1,
 5,
 2,
 10,
 6,
 2,
 -1,
 3,
 -1,
 -1,
 8,
 8,
 8,
 0,
 3,
 8,
 1,
 10,
 7,
 -1,
 1,
 6,
 0,
 8,
 0,
 6,
 2,
 5,
 1,
 2,
 3,
 6,
 1,
 3,
 3,
 6,
 -1,
 -1,
 5,
 3,
 8,
 3,
 4,
 5,
 0,
 7,
 7,
 6,
 3,
 2,
 1,
 -1,
 6,
 0,
 2,
 0,
 -1,
 -1,
 5,
 2,
 -1,
 -1,
 8,
 2,
 1,
 6,
 8,
 -1,
 6,
 3,
 -1,
 7,
 8,
 6,
 6,
 -1,
 6,
 1,
 2,
 3,
 0,
 2,
 2,
 -1,
 4,
 10,
 5,
 9,
 6,
 2,
 10,
 2,
 1,
 6,
 4,
 3,
 -1,
 -1,
 0,
 -1,
 4,
 1,
 1,
 -1,
 1,
 0,
 6,
 8,
 8,
 1,
 1,
 10,
 -1,
 -1,
 5,
 5,
 2,
 5,
 3,
 2,
 4,
 11,
 3,
 6,
 6,
 -1,
 1,
 -1,
 3,
 1,
 2,
 0,
 3,
 0,
 -1,
 -1,
 -1,
 2,
 3,
 6,
 2,
 -1,
 3,
 4,
 7,
 0,
 2,
 8,
 -1,
 4,
 1,
 6,
 -1,
 1,
 -1,
 2,
 -1,
 5,
 6,
 1,
 2,
 5,
 1,
 0,
 1,
 3,
 1,
 -1,
 4,
 5,
 0,
 6,
 -1,
 6,
 1,
 2,
 1,
 -1,
 6,
 -1,
 1,
 9,
 -1,
 -1,
 -1,
 2,
 7,
 -1,
 8,
 4,
 8,
 3,
 -1,
 3,
 1,
 3,
 6,
 -1,
 8,
 10,
 2,
 2,
 -1,
 -1,
 1,
 -1,